Copyright (C) 2026 S.H.Tekur

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>.

In [ ]:
# =====================================================================
# IMPORTS
# =====================================================================
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import random
import numpy as np
from pathlib import Path

import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torchvision import models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, roc_curve
)
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# =====================================================================
# CUSTOM TRANSFORMS
# =====================================================================

class CircularMask:
    def __init__(self, radius_ratio=0.48, jitter=0.0, fill=0.5):
        self.radius_ratio = radius_ratio
        self.jitter = jitter
        self.fill = fill

    def __call__(self, img_tensor):
        _, h, w = img_tensor.shape
        cy = h / 2 + random.uniform(-self.jitter, self.jitter) * h
        cx = w / 2 + random.uniform(-self.jitter, self.jitter) * w
        r  = self.radius_ratio * min(h, w) * random.uniform(0.98, 1.02)
        yy, xx = torch.meshgrid(
            torch.arange(h, device=img_tensor.device),
            torch.arange(w, device=img_tensor.device),
            indexing="ij"
        )
        mask = (((yy - cy)**2 + (xx - cx)**2) <= r**2).float().unsqueeze(0)
        return img_tensor * mask + self.fill * (1.0 - mask)


class AdaptiveThreshold:
    def __init__(self, block_size=31, C=5, fill_bg=0.5):
        self.block_size = block_size
        self.C = C
        self.fill_bg = fill_bg

    def __call__(self, img_tensor):
        img_np = img_tensor.squeeze(0).cpu().numpy()
        img_np = np.clip(img_np * 255.0, 0, 255).astype(np.uint8)
        try:
            binary_np = cv2.adaptiveThreshold(
                img_np, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, self.block_size, self.C
            )
        except Exception:
            _, binary_np = cv2.threshold(
                img_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary_np = cv2.morphologyEx(binary_np, cv2.MORPH_CLOSE, kernel)
        binary_np = cv2.medianBlur(binary_np, 3)
        binary_np = self._thin_lines_fallback(binary_np)
        binary_tensor = torch.from_numpy(binary_np).float().unsqueeze(0) / 255.0
        binary_tensor = torch.where(
            binary_tensor > 0.5,
            torch.tensor(self.fill_bg),
            torch.tensor(0.0)
        )
        return binary_tensor

    def _thin_lines_fallback(self, binary_np):
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        thinned = binary_np.copy()
        for _ in range(10):
            eroded  = cv2.erode(thinned, kernel, iterations=1)
            dilated = cv2.dilate(eroded,  kernel, iterations=1)
            if np.array_equal(thinned, dilated):
                break
            thinned = dilated
        return thinned


In [ ]:
# =====================================================================
# DATASET — deterministic (no augmentation, for eval only)
# =====================================================================

class ChiralFermiDataset(Dataset):
    def __init__(self, root_dir, split, image_size=224):
        self.data_dir = Path(root_dir) / split
        self.image_paths = []
        self.labels = []

        for class_dir in sorted(self.data_dir.iterdir()):
            if class_dir.is_dir() and class_dir.name in ['R', 'S']:
                label = 0 if class_dir.name == 'R' else 1
                for img_path in class_dir.glob('*'):
                    if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                        self.image_paths.append(img_path)
                        self.labels.append(label)

        self.preprocess = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size), antialias=True),
            transforms.ToTensor(),
            CircularMask(radius_ratio=0.48, jitter=0.0, fill=0.5),
            AdaptiveThreshold(block_size=31, C=5, fill_bg=0.5),
        ])

        print(f"{split}: {len(self.image_paths)} images "
              f"(R={self.labels.count(0)}, S={self.labels.count(1)})")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        img = self.preprocess(img)
        img = img.repeat(3, 1, 1)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


In [ ]:
# =====================================================================
# PATHS, DATALOADERS, MODEL
# =====================================================================

ROOT_DIR   = "fermi_surfaces"
MODEL_PATH = "fermi_surface_rs_classifier.pth"

# Datasets
val_dataset  = ChiralFermiDataset(ROOT_DIR, split='val',  image_size=224)
test_dataset = ChiralFermiDataset(ROOT_DIR, split='test', image_size=224)

val_loader  = DataLoader(val_dataset,  batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

# Model — plain Linear head to match saved weights
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

model = build_model().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Model loaded from: {MODEL_PATH}")


In [ ]:
# =====================================================================
# METRICS + CONFUSION MATRIX + ROC CURVE
# =====================================================================

def get_all_preds(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs > 0.5).astype(int)
    return all_labels, all_preds, all_probs


def print_metrics(labels, preds, probs, split_name):
    acc    = accuracy_score(labels, preds)
    prec_r = precision_score(labels, preds, pos_label=0, zero_division=0)
    rec_r  = recall_score(labels, preds, pos_label=0, zero_division=0)
    prec_s = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec_s  = recall_score(labels, preds, pos_label=1, zero_division=0)
    f1     = f1_score(labels, preds, zero_division=0)
    auc    = roc_auc_score(labels, probs)

    print(f"\n{'='*45}")
    print(f"  {split_name}")
    print(f"{'='*45}")
    print(f"  Accuracy          : {acc*100:.2f}%")
    print(f"  Precision (R)     : {prec_r*100:.2f}%")
    print(f"  Recall    (R)     : {rec_r*100:.2f}%")
    print(f"  Precision (S)     : {prec_s*100:.2f}%")
    print(f"  Recall    (S)     : {rec_s*100:.2f}%")
    print(f"  F1 Score          : {f1*100:.2f}%")
    print(f"  AUC-ROC           : {auc:.4f}")
    print(f"{'='*45}\n")

    return dict(split=split_name, accuracy=acc,
                precision_R=prec_r, recall_R=rec_r,
                precision_S=prec_s, recall_S=rec_s,
                f1=f1, auc=auc,
                labels=labels, preds=preds, probs=probs)


# ---- Run ----
val_labels,  val_preds,  val_probs  = get_all_preds(model, val_loader,  device)
test_labels, test_preds, test_probs = get_all_preds(model, test_loader, device)

val_results  = print_metrics(val_labels,  val_preds,  val_probs,  "Validation Set")
test_results = print_metrics(test_labels, test_preds, test_probs, "Test Set")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. EDIT TYPOGRAPHY HERE
# =========================================================
FONT_FAMILY = 'Helvetica'          
BASE_SIZE = 14

TITLE_SIZE = 17
AXIS_LABEL_SIZE = 14
TICK_SIZE = 13
NUMBER_SIZE = 14
CAPTION_SIZE = 14

TITLE_WEIGHT = 'normal'
AXIS_LABEL_WEIGHT = 'normal'
TICK_WEIGHT = 'normal'
NUMBER_WEIGHT = 'normal'
CAPTION_WEIGHT = 'bold'

TITLE_STYLE = 'normal'
AXIS_LABEL_STYLE = 'normal'
TICK_STYLE = 'normal'
NUMBER_STYLE = 'normal'
CAPTION_STYLE = 'normal'

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['font.size'] = BASE_SIZE
plt.rcParams['svg.fonttype'] = 'none'   

# =========================================================
# 2. ENTER YOUR DATA HERE
# Rows = true labels, columns = predicted labels
# =========================================================
cm_fermi = np.array([
    [100.00, 0.00],
    [1.74, 98.26]
])

class_labels = ['R', 'S']
title_a = "Test Set\nAcc=99.13%  AUC=1.000"

# =========================================================
# 3. FIGURE
# =========================================================
fig, ax = plt.subplots(figsize=(5.4, 5.8), constrained_layout=True)

im = ax.imshow(cm_fermi, cmap='Blues', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(class_labels)))
ax.set_yticks(np.arange(len(class_labels)))
ax.set_xticklabels(class_labels)
ax.set_yticklabels(class_labels)

ax.set_xlabel(
    "Predicted label",
    fontsize=AXIS_LABEL_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=AXIS_LABEL_WEIGHT,
    fontstyle=AXIS_LABEL_STYLE
)
ax.set_ylabel(
    "True label",
    fontsize=AXIS_LABEL_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=AXIS_LABEL_WEIGHT,
    fontstyle=AXIS_LABEL_STYLE
)
ax.set_title(
    title_a,
    fontsize=TITLE_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=TITLE_WEIGHT,
    fontstyle=TITLE_STYLE,
    pad=10
)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontsize(TICK_SIZE)
    label.set_fontfamily(FONT_FAMILY)
    label.set_fontweight(TICK_WEIGHT)
    label.set_fontstyle(TICK_STYLE)

for i in range(cm_fermi.shape[0]):
    for j in range(cm_fermi.shape[1]):
        color = 'white' if cm_fermi[i, j] > 50 else '#1f3b73'
        ax.text(
            j, i, f"{cm_fermi[i, j]:.2f}%",
            ha='center', va='center',
            color=color,
            fontsize=NUMBER_SIZE,
            fontfamily=FONT_FAMILY,
            fontweight=NUMBER_WEIGHT,
            fontstyle=NUMBER_STYLE
        )

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

ax.text(
    0.5, -0.16,
    "(a) Confusion matrix for Fermi surface classifier",
    transform=ax.transAxes,
    ha='center', va='top',
    fontsize=CAPTION_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=CAPTION_WEIGHT,
    fontstyle=CAPTION_STYLE
)

# =========================================================
# 4. SAVE
# =========================================================
plt.savefig("fermisurface_confusion_matrix.png", dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.transforms import ScaledTranslation

# =========================================================
# 1. TYPOGRAPHY / STYLE
# =========================================================
FONT_FAMILY = 'Helvetica'   # fallback to Arial/DejaVu Sans if unavailable
BASE_SIZE = 20

TITLE_SIZE = 20
AXIS_LABEL_SIZE = 20
TICK_SIZE = 20
NUMBER_SIZE = 20
PANEL_LABEL_SIZE = 20

TITLE_WEIGHT = 'normal'
AXIS_LABEL_WEIGHT = 'normal'
TICK_WEIGHT = 'normal'
NUMBER_WEIGHT = 'normal'
PANEL_LABEL_WEIGHT = 'bold'

TITLE_STYLE = 'normal'
AXIS_LABEL_STYLE = 'normal'
TICK_STYLE = 'normal'
NUMBER_STYLE = 'normal'
PANEL_LABEL_STYLE = 'normal'

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['font.size'] = BASE_SIZE
plt.rcParams['svg.fonttype'] = 'none'

# =========================================================
# 2. DATA
# =========================================================
# Panel (b): confusion matrix
cm_real = np.array([
    [100.00, 0.00],
    [1.74, 98.26]
])
class_labels = ['R', 'S']
title_b = "Test Set (Acc=99.1%)"

# Panel (c): performance metrics
metric_names = [
    'Accuracy',
    'Precision (R)',
    'Recall (R)',
    'Precision (S)',
    'Recall (S)',
    'F1 Score',
    'AUC-ROC'
]

metric_values = [
    99.13,
    98.29,
    100.00,
    100.00,
    98.26,
    99.12,
    100.00
]

# =========================================================
# 3. FIGURE — POWERPOINT WIDESCREEN (16:9)
# =========================================================
fig, axes = plt.subplots(
    1, 2,
    figsize=(13.33, 7.5),
    gridspec_kw={'width_ratios': [1.0, 1.7]},
    constrained_layout=True
)

# A little extra separation between panels
fig.set_constrained_layout_pads(w_pad=3/72, h_pad=3/72, wspace=0.08, hspace=0.02)

# =========================================================
# 4. PANEL (a): CONFUSION MATRIX
# =========================================================
ax = axes[0]
im = ax.imshow(cm_real, cmap='Blues', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(class_labels)))
ax.set_yticks(np.arange(len(class_labels)))
ax.set_xticklabels(class_labels)
ax.set_yticklabels(class_labels)

ax.set_xlabel(
    "Predicted label",
    fontsize=AXIS_LABEL_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=AXIS_LABEL_WEIGHT,
    fontstyle=AXIS_LABEL_STYLE,
    labelpad=10
)
ax.set_ylabel(
    "True label",
    fontsize=AXIS_LABEL_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=AXIS_LABEL_WEIGHT,
    fontstyle=AXIS_LABEL_STYLE,
    labelpad=10
)
ax.set_title(
    title_b,
    fontsize=TITLE_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=TITLE_WEIGHT,
    fontstyle=TITLE_STYLE,
    pad=12
)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontsize(TICK_SIZE)
    label.set_fontfamily(FONT_FAMILY)
    label.set_fontweight(TICK_WEIGHT)
    label.set_fontstyle(TICK_STYLE)

for i in range(cm_real.shape[0]):
    for j in range(cm_real.shape[1]):
        color = 'white' if cm_real[i, j] > 50 else '#1f3b73'
        ax.text(
            j, i, f"{cm_real[i, j]:.2f}%",
            ha='center', va='center',
            color=color,
            fontsize=NUMBER_SIZE,
            fontfamily=FONT_FAMILY,
            fontweight=NUMBER_WEIGHT,
            fontstyle=NUMBER_STYLE
        )

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

# # Panel label outside axes, top-left
# ax.text(
#     0.0, 1.0, "(a)",
#     transform=ax.transAxes + ScaledTranslation(-34/72, 10/72, fig.dpi_scale_trans),
#     ha='right', va='bottom',
#     fontsize=PANEL_LABEL_SIZE,
#     fontfamily=FONT_FAMILY,
#     fontweight=PANEL_LABEL_WEIGHT,
#     fontstyle=PANEL_LABEL_STYLE
# )

# =========================================================
# 5. PANEL (b): HORIZONTAL BAR CHART
# =========================================================
ax = axes[1]
y = np.arange(len(metric_names))

bars = ax.barh(
    y, metric_values,
    color='#4C78A8',
    edgecolor='black',
    linewidth=0.8,
    alpha=0.92,
    height=0.72
)

ax.set_yticks(y)
ax.set_yticklabels(metric_names)
ax.invert_yaxis()

# give some breathing room for labels at bar ends
ax.set_xlim(97, 100.5)
ax.margins(x=0.02)

ax.set_xlabel(
    "Score (%)",
    fontsize=AXIS_LABEL_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=AXIS_LABEL_WEIGHT,
    fontstyle=AXIS_LABEL_STYLE,
    labelpad=10
)
ax.set_title(
    "Classifier performance",
    fontsize=TITLE_SIZE,
    fontfamily=FONT_FAMILY,
    fontweight=TITLE_WEIGHT,
    fontstyle=TITLE_STYLE,
    pad=12
)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontsize(TICK_SIZE)
    label.set_fontfamily(FONT_FAMILY)
    label.set_fontweight(TICK_WEIGHT)
    label.set_fontstyle(TICK_STYLE)

ax.grid(axis='x', linestyle='-', linewidth=0.4, alpha=0.35)
ax.set_axisbelow(True)

# Clean spines a bit for paper look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.8)
ax.spines['bottom'].set_linewidth(0.8)

for bar, value in zip(bars, metric_values):
    ax.text(
        value + 0.18,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        va='center', ha='left',
        fontsize=NUMBER_SIZE,
        fontfamily=FONT_FAMILY,
        fontweight=NUMBER_WEIGHT,
        fontstyle=NUMBER_STYLE
    )

# # Panel label outside axes, top-left
# ax.text(
#     0.0, 1.0, "(b)",
#     transform=ax.transAxes + ScaledTranslation(-34/72, 10/72, fig.dpi_scale_trans),
#     ha='right', va='bottom',
#     fontsize=PANEL_LABEL_SIZE,
#     fontfamily=FONT_FAMILY,
#     fontweight=PANEL_LABEL_WEIGHT,
#     fontstyle=PANEL_LABEL_STYLE
# )

# =========================================================
# 6. SAVE
# 13.33 x 7.5 at 144 dpi ≈ 1920 x 1080 px
# =========================================================
plt.savefig(
    "fermi_surface_classifier_performance.png",
    dpi=144
)

plt.show()